# Random Forest para depressão (NHIS 2025): binário e multinível, com e sem ansiedade


## Hipótese

> É possível prever depressão a partir de dados de saúde física e do contexto de vida, **sem** depender de sintomas de transtornos mentais.

Esperamos que o desempenho sem sintomas mentais seja **pior** do que com eles. A hipótese não exige que sejam iguais: ela se sustenta se o modelo ainda tiver poder preditivo real, bem acima do acaso.

## Cenários

1. **Cenário A (exploratório, com ansiedade):** removemos de $X$ apenas os sintomas de depressão (PHQ) e **mantemos** as variáveis de ansiedade (GAD-7). Serve para explorar os dados e como referência de desempenho máximo.
2. **Cenário B (hipótese, sem sintomas mentais):** removemos de $X$ os sintomas de depressão **e** de ansiedade. Sobram variáveis sociodemográficas, de saúde física, de acesso a serviços de saúde e de contexto social.

## Roteiro

1. Preparação dos dados e definição dos cenários.
2. Função de treino e avaliação (alvo binário).
3. Cenário A e Cenário B, comparativo e variáveis mais importantes.
4. **Por que usar o alvo binário** (comparação com os 4 níveis de severidade do PHQ).
5. Ajuste de hiperparâmetros (`RandomizedSearchCV`) no cenário B, binário e multinível.
6. Perfil médio por nível de severidade.
7. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?
8. Interpretação e conclusão.

## Desenho do experimento (para a comparação ser justa)

- Mesma base (`adult25_limpo.csv`), mesmo `PESO_AMOSTRAL` e mesmo split treino/teste (80/20 estratificado, `random_state=42`) em todos os modelos.
- **Mesmas linhas** nos dois cenários: registros sem `-1` (não resposta) nas variáveis do cenário A, o mais restritivo. Nos notebooks antigos, manter ou remover o `-1` de $X$ mudou pouco o resultado (acurácia de 0,77 contra 0,78 no multinível), por isso essa escolha foi unificada: usamos só registros completos.
- Mesmo modelo e mesmos hiperparâmetros-base nos dois cenários. Só muda o conjunto de colunas.

# 1. Preparação dos dados e definição dos cenários

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

PESO_COLUNA = 'PESO_AMOSTRAL'
TARGET_ORIGINAL = 'SEVERIDADE_DEPRESSAO_PHQ'
TARGET_BIN = 'DEPRESSAO_PHQ_SIMPLIFICADO'
RANDOM_STATE = 42

# Sintomas diretos de depressão (PHQ): removidos nos DOIS cenários
COLUNAS_DEPRESSAO = [
    'FREQ_PERDA_INTERESSE_2SEM', 'FREQ_SENTIU_DEPRIMIDO_2SEM',
    'FREQ_DIFICULDADE_DORMIR_2SEM', 'FREQ_CANSASO_2SEM',
    'FREQ_HIPOREXIA_COMPULSAO_ALIMENTAR_2SEM', 'FREQ_AUTOAVALIACAO_NEGTIVA_2SEM',
    'FREQ_DIFICULDADE_CONCENTRACAO_2SEM', 'MODIFICACAO_VELOCIDADE_FALA',
    'FREQ_DEPRESSAO', 'REMEDIO_DEPRESSAO', 'HISTORICO_DEPRESSAO',
    'INTENSIDADE_DEPRESSAO', 'SEVERIDADE_DEPRESSAO_PHQ'
]

# Ansiedade (GAD-7): mantida no cenário A, removida no cenário B
COLUNAS_ANSIEDADE = [
    'HISTORICO_TRANSTORNO_ANSIEDADE', 'MED_ANSIEDADE',
    'FREQ_ANSIEDADE', 'INTENSIDADE_ANSIEDADE',
    'FREQ_ANSIEDADE_2SEM', 'FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM',
    'FREQ_PREOCUPACAO_DESNECESSARIA_2SEM', 'FREQ_DIFICULDADE_DESCANSAR_2SEM',
    'FREQ_AGITACAO_INQUIETA', 'FREQ_IRRITABILIDADE_2SEM',
    'FREQ_MEDO_2SEM', 'SEVERIDADE_ANSIEDADE'
]

# Carregar a base limpa
df = pd.read_csv(Path('../../01 Preparação dos dados/dados/limpos') / 'adult25_limpo.csv').set_index('ID_DOMICILIO')

# Garante que nenhuma coluna foi digitada errado (o drop abaixo ignoraria erros silenciosamente)
ausentes = [c for c in COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE if c not in df.columns]
assert not ausentes, f'Colunas inexistentes na base: {ausentes}'

# Remover registros indefinidos no alvo (-1 e 5) e binarizar: 0 = sem depressão | 1 = com depressão
df = df[~df[TARGET_ORIGINAL].isin([-1, 5])].copy()
df[TARGET_BIN] = (df[TARGET_ORIGINAL] != 1).astype(int)

# Definição dos dois cenários: o que é removido de X em cada um
CENARIOS = {
    'A: com ansiedade (exploratório)': COLUNAS_DEPRESSAO,
    'B: sem sintomas mentais (hipótese)': COLUNAS_DEPRESSAO + COLUNAS_ANSIEDADE,
}
BASE_REMOVER = [TARGET_BIN, PESO_COLUNA]

# Mesmas linhas nos dois cenários: registros sem -1 nas variáveis do cenário A (o mais restritivo)
X_cenario_a = df.drop(columns=BASE_REMOVER + CENARIOS['A: com ansiedade (exploratório)'])
mascara_completos = (X_cenario_a != -1).all(axis=1).to_numpy()
df_modelo = df[mascara_completos].copy()

# Mesmo split nos dois cenários (por posição)
posicoes = np.arange(len(df_modelo))
pos_tr, pos_te = train_test_split(
    posicoes, test_size=0.2, random_state=RANDOM_STATE, stratify=df_modelo[TARGET_BIN]
)

y_tr = df_modelo[TARGET_BIN].iloc[pos_tr]
y_te = df_modelo[TARGET_BIN].iloc[pos_te]
w_tr = df_modelo[PESO_COLUNA].iloc[pos_tr]
w_te = df_modelo[PESO_COLUNA].iloc[pos_te]

print(f'Registros no experimento: {len(df_modelo)} (treino: {len(pos_tr)} | teste: {len(pos_te)})')
print(f'Proporção de depressão (ponderada pelo peso amostral): {np.average(df_modelo[TARGET_BIN], weights=df_modelo[PESO_COLUNA]):.1%}')
for nome, remover in CENARIOS.items():
    n_feat = df_modelo.drop(columns=BASE_REMOVER + remover).shape[1]
    print(f'Cenário {nome}: {n_feat} variáveis preditoras')

Registros no experimento: 22084 (treino: 17667 | teste: 4417)
Proporção de depressão (ponderada pelo peso amostral): 21.2%
Cenário A: com ansiedade (exploratório): 58 variáveis preditoras
Cenário B: sem sintomas mentais (hipótese): 46 variáveis preditoras


# 2. Função de treino e avaliação

A mesma função é usada nos dois cenários, garantindo que a **única diferença** entre eles seja o conjunto de variáveis. Retorna as métricas da classe positiva (com depressão), ponderadas pelo `PESO_AMOSTRAL`, a matriz de confusão em **contagem de registros reais** do teste (sem peso), a matriz ponderada (usada só na análise final) e a importância das variáveis do modelo.

In [2]:
def treinar_avaliar(colunas_remover):
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]

    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    X_tr_proc = preprocessor.fit_transform(X_tr)
    X_te_proc = preprocessor.transform(X_te)

    modelo = RandomForestClassifier(
        n_estimators=300, max_depth=25, min_samples_split=5,
        max_features='log2', class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    modelo.fit(X_tr_proc, y_tr, sample_weight=w_tr)

    y_pred = modelo.predict(X_te_proc)
    y_proba = modelo.predict_proba(X_te_proc)[:, 1]

    metricas = {
        'Nº de variáveis': X.shape[1],
        'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
        'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
        'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
        'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    }
    matriz = confusion_matrix(y_te, y_pred)
    matriz_pond = np.round(confusion_matrix(y_te, y_pred, sample_weight=w_te)).astype(int)

    # Importância agregada por variável original (soma das colunas One-Hot de cada uma)
    nomes = preprocessor.get_feature_names_out()
    importancias = pd.Series(modelo.feature_importances_, index=nomes)
    originais = sorted(X.columns, key=len, reverse=True)
    agregada = {}
    for nome_col, imp in importancias.items():
        base = nome_col.split('__', 1)[1]
        dono = next((c for c in originais if base == c or base.startswith(c + '_')), base)
        agregada[dono] = agregada.get(dono, 0) + imp
    importancia_var = pd.Series(agregada).sort_values(ascending=False)

    return metricas, matriz, matriz_pond, importancia_var

resultados = {}

# 3. Cenário A: com variáveis de ansiedade (exploratório)

Removemos de $X$ somente os sintomas de depressão (PHQ). As variáveis de ansiedade (GAD-7) permanecem. Como ansiedade e depressão são fortemente comórbidas, este cenário mostra o **melhor desempenho que conseguimos quando há informação de saúde mental disponível**.

In [3]:
nome_a = 'A: com ansiedade (exploratório)'
resultados[nome_a] = treinar_avaliar(CENARIOS[nome_a])
metricas_a, matriz_a, matriz_pond_a, imp_a = resultados[nome_a]

print(f'--- {nome_a} ---')
for k, v in metricas_a.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_a)
print('\nTop 10 variáveis mais importantes:')
print(imp_a.head(10).round(4).to_string())

--- A: com ansiedade (exploratório) ---
Nº de variáveis: 58
ROC-AUC: 0.9135
Acurácia: 0.8606
Precisão (depressão): 0.6556
Recall (depressão): 0.7491
F1 (depressão): 0.6993

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3113  369]
 [ 229  706]]

Top 10 variáveis mais importantes:
TIPO_DIFICULDADE_COGNITIVA             0.0607
SEVERIDADE_ANSIEDADE                   0.0598
FREQ_ANSIEDADE                         0.0571
IDADE                                  0.0516
FREQ_ANSIEDADE_2SEM                    0.0469
FREQ_PREOCUPACAO_DESNECESSARIA_2SEM    0.0458
FREQ_PREOCUPACAO_INCONTROLAVEL_2SEM    0.0447
FREQ_DIFICULDADE_DESCANSAR_2SEM        0.0399
FREQ_IRRITABILIDADE_2SEM               0.0339
FREQ_MEDO_2SEM                         0.0302


# 4. Cenário B: sem sintomas de saúde mental (hipótese do projeto)

Removemos de $X$ os sintomas de depressão **e** de ansiedade. O modelo passa a usar apenas variáveis sociodemográficas, de saúde física (doenças e condições crônicas), de acesso a serviços de saúde e de contexto social. É aqui que testamos se dá para **prever depressão a partir de doenças físicas e do contexto de vida**.

In [4]:
nome_b = 'B: sem sintomas mentais (hipótese)'
resultados[nome_b] = treinar_avaliar(CENARIOS[nome_b])
metricas_b, matriz_b, matriz_pond_b, imp_b = resultados[nome_b]

print(f'--- {nome_b} ---')
for k, v in metricas_b.items():
    print(f'{k}: {v:.4f}' if isinstance(v, float) else f'{k}: {v}')
print('\nMatriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):')
print(matriz_b)
print('\nTop 10 variáveis mais importantes:')
print(imp_b.head(10).round(4).to_string())

--- B: sem sintomas mentais (hipótese) ---
Nº de variáveis: 46
ROC-AUC: 0.8411
Acurácia: 0.8153
Precisão (depressão): 0.5713
Recall (depressão): 0.5857
F1 (depressão): 0.5784

Matriz de confusão (nº de registros do teste; linhas = real, colunas = previsto):
[[3017  465]
 [ 350  585]]

Top 10 variáveis mais importantes:
IDADE                          0.0937
TIPO_DIFICULDADE_COGNITIVA     0.0920
FREQ_SOLITARIO                 0.0716
SUPORTE_SOCIAL                 0.0579
QTD_DIFICULDADE_COGNITIVA      0.0476
AUTOAVALIACAO_SAUDE_GERAL      0.0457
FREQUENCIA_FALHA_MEMORIA       0.0452
CATEGORIA_RAZAO_POBREZA        0.0436
DOR_CRONICA                    0.0409
ESCOLARIDADE_MAXIMA_FAMILIA    0.0346


# 5. Comparativo entre os cenários

A coluna **Variação (B − A)** mostra quanto desempenho se perde ao retirar a ansiedade. A coluna **Referência (acaso)** mostra o que um modelo sem nenhuma informação conseguiria: ROC-AUC de 0,5 e precisão igual à prevalência de depressão.

In [5]:
prevalencia = np.average(y_te, weights=w_te)

tabela = pd.DataFrame({nome: res[0] for nome, res in resultados.items()})
tabela['Variação (B − A)'] = tabela[nome_b] - tabela[nome_a]
tabela['Referência (acaso)'] = [np.nan, 0.5, np.nan, prevalencia, np.nan, np.nan]

print(f'Prevalência de depressão no teste (ponderada): {prevalencia:.1%}\n')
tabela.round(4)

Prevalência de depressão no teste (ponderada): 21.6%



,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese),Variação (B − A),Referência (acaso)
Nº de variáveis,58.0000,46.0000,-12.0000,NaN
ROC-AUC,0.9135,0.8411,-0.0724,0.5000
Acurácia,0.8606,0.8153,-0.0453,NaN
Precisão (depressão),0.6556,0.5713,-0.0843,0.2163
Recall (depressão),0.7491,0.5857,-0.1635,NaN
F1 (depressão),0.6993,0.5784,-0.1209,NaN


In [6]:
# Variáveis que o cenário B (sem sintomas mentais) usa para prever depressão
print('Top 15 variáveis do cenário B (importância agregada por variável original):\n')
print(imp_b.head(15).round(4).to_string())

ganho_auc = metricas_b['ROC-AUC'] - 0.5
perda_auc = metricas_a['ROC-AUC'] - metricas_b['ROC-AUC']
print(f"\nROC-AUC do cenário B acima do acaso: +{ganho_auc:.3f}")
print(f"ROC-AUC perdido ao retirar a ansiedade (A → B): -{perda_auc:.3f}")

# Matrizes ponderadas pelo PESO_AMOSTRAL: projeção para a população, útil só para a análise final
print('\nMatrizes de confusão ponderadas (projeção populacional; linhas = real, colunas = previsto):')
for nome, mp in [(nome_a, matriz_pond_a), (nome_b, matriz_pond_b)]:
    print(f'\n{nome}')
    print(mp)

Top 15 variáveis do cenário B (importância agregada por variável original):

IDADE                           0.0937
TIPO_DIFICULDADE_COGNITIVA      0.0920
FREQ_SOLITARIO                  0.0716
SUPORTE_SOCIAL                  0.0579
QTD_DIFICULDADE_COGNITIVA       0.0476
AUTOAVALIACAO_SAUDE_GERAL       0.0457
FREQUENCIA_FALHA_MEMORIA        0.0452
CATEGORIA_RAZAO_POBREZA         0.0436
DOR_CRONICA                     0.0409
ESCOLARIDADE_MAXIMA_FAMILIA     0.0346
RAZAO_RENDA_POBREZA             0.0338
TEMPO_MORADIA_ANOS              0.0240
LIMITACAO_TRABALHO_POR_SAUDE    0.0225
ESTRESSE_FINANCEIRO_MEDICO      0.0224
CATEGORIA_IMC                   0.0215

ROC-AUC do cenário B acima do acaso: +0.341
ROC-AUC perdido ao retirar a ansiedade (A → B): -0.072

Matrizes de confusão ponderadas (projeção populacional; linhas = real, colunas = previsto):

A: com ansiedade (exploratório)
[[33671240  4101965]
 [ 2615251  7810026]]

B: sem sintomas mentais (hipótese)
[[33192095  4581109]
 [ 4319371  

# 6. Por que usar o alvo binário?

O PHQ original tem 4 níveis de severidade (1 = mínima, 2 = leve, 3 = moderada, 4 = grave). O alvo binário agrupa os níveis 2 a 4 como **com depressão** e deixa o nível 1 como **sem depressão**. Para justificar essa escolha, treinamos o mesmo Random Forest para prever os 4 níveis, nas mesmas linhas e no mesmo split, e comparamos.

In [7]:
y4 = df_modelo[TARGET_ORIGINAL]
y4_tr, y4_te = y4.iloc[pos_tr], y4.iloc[pos_te]

# Tamanho de cada nível: quantidade de registros e proporção ponderada pela amostra
distribuicao = pd.DataFrame({
    'Registros': y4.value_counts().sort_index(),
    '% ponderado': (df_modelo.groupby(TARGET_ORIGINAL)[PESO_COLUNA].sum() / df_modelo[PESO_COLUNA].sum() * 100).round(1),
})
distribuicao.index.name = 'Nível PHQ'
distribuicao

,Registros,% ponderado
Nível PHQ,,
1,17407,78.8
2,3139,14.2
3,1012,4.6
4,526,2.4


In [8]:
def preparar_matrizes(colunas_remover):
    X = df_modelo.drop(columns=BASE_REMOVER + colunas_remover)
    X_tr, X_te = X.iloc[pos_tr], X.iloc[pos_te]
    num_cols = [c for c in ['RAZAO_RENDA_POBREZA'] if c in X.columns]
    cat_cols = [c for c in X.columns if c not in num_cols]
    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols),
    ])
    return preprocessor.fit_transform(X_tr), preprocessor.transform(X_te)


def novo_rf(**parametros):
    base = dict(n_estimators=300, max_depth=25, min_samples_split=5, max_features='log2',
                class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    base.update(parametros)
    return RandomForestClassifier(**base)


def avaliar_multinivel(modelo, X_te_proc):
    y_pred = modelo.predict(X_te_proc)
    y_proba = modelo.predict_proba(X_te_proc)
    recall_nivel = recall_score(y4_te, y_pred, average=None, sample_weight=w_te, zero_division=0)
    resumo = {
        'ROC-AUC (OvR)': roc_auc_score(y4_te, y_proba, multi_class='ovr', sample_weight=w_te),
        'Acurácia': accuracy_score(y4_te, y_pred, sample_weight=w_te),
        'F1 macro': f1_score(y4_te, y_pred, average='macro', sample_weight=w_te, zero_division=0),
    }
    for nivel, r in zip(modelo.classes_, recall_nivel):
        resumo[f'Recall nível {nivel}'] = r
    return resumo, confusion_matrix(y4_te, y_pred)


matrizes = {nome: preparar_matrizes(remover) for nome, remover in CENARIOS.items()}
multinivel = {}
for nome, (X_tr_p, X_te_p) in matrizes.items():
    modelo = novo_rf().fit(X_tr_p, y4_tr, sample_weight=w_tr)
    multinivel[nome] = avaliar_multinivel(modelo, X_te_p)

print('Matriz de confusão do cenário B, 4 níveis (nº de registros do teste; linhas = real, colunas = previsto):')
print(multinivel[nome_b][1])
pd.DataFrame({nome: res[0] for nome, res in multinivel.items()}).round(4)

Matriz de confusão do cenário B, 4 níveis (nº de registros do teste; linhas = real, colunas = previsto):
[[2935  516   20   11]
 [ 275  282   30   26]
 [  52  104   30   27]
 [  14   47   10   38]]


,A: com ansiedade (exploratório),B: sem sintomas mentais (hipótese)
ROC-AUC (OvR),0.9062,0.8255
Acurácia,0.7900,0.7498
F1 macro,0.5375,0.4471
Recall nível 1,0.8694,0.8603
Recall nível 2,0.5960,0.4299
Recall nível 3,0.2156,0.1347
Recall nível 4,0.5540,0.3295


### Resultado: os 4 níveis não são separáveis com os dados disponíveis

**1. Os níveis mais graves são raros.** Depois de remover registros incompletos, os níveis 3 e 4 somam 1.538 registros (cerca de 7% da população ponderada), contra 17.407 do nível 1. Com tão poucos exemplos, o modelo tem pouco para aprender sobre eles.

**2. O modelo erra sobretudo nos níveis intermediários.** No cenário B, o recall é de 86% no nível 1, mas só 43% no nível 2, **13% no nível 3** e 33% no nível 4. Na matriz de confusão, dos casos reais de nível 3 apenas 30 são acertados; a maioria vai para os níveis 1 e 2. O F1 macro, que dá o mesmo peso a cada nível, cai para 0,45 (0,54 no cenário A).

**3. A distinção entre níveis vizinhos é pouco informativa.** Os níveis são ordinais e a diferença entre "leve" e "moderado" depende de poucos pontos na escala. Variáveis de saúde física e de contexto social indicam bem **se há** sofrimento depressivo, mas não **quanto**.

**4. O alvo binário responde melhor à pergunta do projeto.** A hipótese é verificar se dá para **identificar** quem tem depressão sem sintomas mentais, e isso é uma pergunta de triagem (sim ou não). O binário tem classes mais equilibradas (cerca de 21% de positivos), métricas diretas (precisão, recall e ROC-AUC) e resultado fácil de interpretar. Mesmo assim, a ordenação por risco é preservada: o ROC-AUC do binário no cenário B (0,84) é equivalente ao do multinível (0,83 em média entre os níveis).

O multinível continua no notebook como **análise complementar**. Ele mostra até onde os dados permitem ir e quais níveis são mais difíceis de prever.

# 7. Ajuste de hiperparâmetros (cenário B)

Até aqui usamos hiperparâmetros fixos. Aqui buscamos uma configuração melhor **apenas no cenário B**, que é o que testa a hipótese, com `RandomizedSearchCV` (15 combinações, validação cruzada de 3 partes). O `PESO_AMOSTRAL` é usado no ajuste do modelo, mas a métrica de seleção da busca não é ponderada. O conjunto de teste não participa da busca.

In [9]:
from sklearn.model_selection import RandomizedSearchCV

ESPACO_BUSCA = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 15, 20, 25, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': ['sqrt', 'log2', 0.3],
    'class_weight': ['balanced', 'balanced_subsample'],
}
X_tr_b, X_te_b = matrizes[nome_b]


def buscar(y_treino, scoring):
    busca = RandomizedSearchCV(
        RandomForestClassifier(random_state=RANDOM_STATE), ESPACO_BUSCA,
        n_iter=15, scoring=scoring, cv=3, random_state=RANDOM_STATE, n_jobs=-1,
    )
    busca.fit(X_tr_b, y_treino, sample_weight=w_tr)
    return busca


# Binário: seleciona pelo ROC-AUC
busca_bin = buscar(y_tr, 'roc_auc')
print('Melhores hiperparâmetros (binário):', busca_bin.best_params_)

modelo_bin = busca_bin.best_estimator_
y_pred = modelo_bin.predict(X_te_b)
y_proba = modelo_bin.predict_proba(X_te_b)[:, 1]
tuned_bin = {
    'ROC-AUC': roc_auc_score(y_te, y_proba, sample_weight=w_te),
    'Acurácia': accuracy_score(y_te, y_pred, sample_weight=w_te),
    'Precisão (depressão)': precision_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    'Recall (depressão)': recall_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
    'F1 (depressão)': f1_score(y_te, y_pred, sample_weight=w_te, zero_division=0),
}
comparacao_bin = pd.DataFrame({
    'B base': {k: resultados[nome_b][0][k] for k in tuned_bin},
    'B ajustado': tuned_bin,
})
comparacao_bin['Variação'] = comparacao_bin['B ajustado'] - comparacao_bin['B base']
print('\nMatriz de confusão do modelo ajustado (nº de registros do teste):')
print(confusion_matrix(y_te, y_pred))
comparacao_bin.round(4)

Melhores hiperparâmetros (binário): {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 25, 'class_weight': 'balanced'}

Matriz de confusão do modelo ajustado (nº de registros do teste):
[[3017  465]
 [ 350  585]]


,B base,B ajustado,Variação
ROC-AUC,0.8411,0.8411,0.0
Acurácia,0.8153,0.8153,0.0
Precisão (depressão),0.5713,0.5713,0.0
Recall (depressão),0.5857,0.5857,0.0
F1 (depressão),0.5784,0.5784,0.0


In [10]:
# Multinível (4 níveis): seleciona pelo F1 macro, que dá o mesmo peso a cada nível
busca_mult = buscar(y4_tr, 'f1_macro')
print('Melhores hiperparâmetros (4 níveis):', busca_mult.best_params_)

modelo_mult = busca_mult.best_estimator_
res_ajustado, matriz_ajustada = avaliar_multinivel(modelo_mult, X_te_b)
print('\nMatriz de confusão do modelo ajustado, 4 níveis (nº de registros do teste):')
print(matriz_ajustada)
comparacao_mult = pd.DataFrame({'B base': multinivel[nome_b][0], 'B ajustado': res_ajustado})
comparacao_mult['Variação'] = comparacao_mult['B ajustado'] - comparacao_mult['B base']
comparacao_mult.round(4)

Melhores hiperparâmetros (4 níveis): {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_features': 0.3, 'max_depth': None, 'class_weight': 'balanced_subsample'}

Matriz de confusão do modelo ajustado, 4 níveis (nº de registros do teste):
[[2814  574   64   30]
 [ 231  268   69   45]
 [  42   84   41   46]
 [   9   29   25   46]]


,B base,B ajustado,Variação
ROC-AUC (OvR),0.8255,0.8248,-0.0008
Acurácia,0.7498,0.7296,-0.0202
F1 macro,0.4471,0.4399,-0.0071
Recall nível 1,0.8603,0.8278,-0.0326
Recall nível 2,0.4299,0.4345,0.0046
Recall nível 3,0.1347,0.1871,0.0524
Recall nível 4,0.3295,0.4096,0.0801


# 8. Perfil médio por nível de severidade

Para entender o que distingue os níveis do PHQ, comparamos a média (ponderada pela amostra) das variáveis mais importantes do cenário B em cada nível. As variáveis são códigos categóricos ordinais, então a média serve apenas para mostrar a **tendência** entre os níveis, e não um valor literal. Os valores `-1` (não resposta) já estão fora desta base.

In [11]:
top_b = [c for c in resultados[nome_b][3].head(8).index if c != 'RAZAO_RENDA_POBREZA']


def media_ponderada(grupo):
    return pd.Series({c: np.average(grupo[c], weights=grupo[PESO_COLUNA]) for c in top_b})


perfil = df_modelo.groupby(TARGET_ORIGINAL)[[*top_b, PESO_COLUNA]].apply(media_ponderada)
perfil.index.name = 'Nível PHQ'
perfil.round(2)

,IDADE,TIPO_DIFICULDADE_COGNITIVA,FREQ_SOLITARIO,SUPORTE_SOCIAL,QTD_DIFICULDADE_COGNITIVA,AUTOAVALIACAO_SAUDE_GERAL,FREQUENCIA_FALHA_MEMORIA,CATEGORIA_RAZAO_POBREZA
Nível PHQ,,,,,,,,
1,48.96,3.66,1.61,4.38,1.17,3.77,3.61,10.16
2,46.24,3.17,2.38,3.97,1.54,3.18,2.93,9.16
3,43.76,2.97,2.82,3.59,1.81,2.96,2.69,8.62
4,46.77,2.89,3.41,3.28,2.13,2.36,2.51,7.31


# 9. Teste de sensibilidade: o resultado depende das variáveis cognitivas e de solidão?

No cenário B, `TIPO_DIFICULDADE_COGNITIVA` está entre as variáveis mais importantes. Ela vem da pergunta sobre dificuldade para **lembrar ou se concentrar**, e dificuldade de concentração também é um item do PHQ, que já removemos. Há o risco de essa variável reintroduzir sintoma depressivo no modelo e inflar o resultado. O mesmo cuidado vale para a solidão.

Para testar, repetimos o cenário B (mesmas linhas, mesmo split e mesmos hiperparâmetros) retirando progressivamente:

1. `TIPO_DIFICULDADE_COGNITIVA` sozinha;
2. o **bloco cognitivo** inteiro, que vem da mesma pergunta (`TIPO_DIFICULDADE_COGNITIVA`, `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA`);
3. o bloco cognitivo **e** `FREQ_SOLITARIO`.

O teste de retirar só uma variável é fraco: as outras do bloco carregam a mesma informação e assumem o lugar dela. Por isso o passo 2 é o mais informativo.

In [ ]:
BLOCO_COGNITIVO = ['TIPO_DIFICULDADE_COGNITIVA', 'QTD_DIFICULDADE_COGNITIVA', 'FREQUENCIA_FALHA_MEMORIA']
SOLIDAO = ['FREQ_SOLITARIO']
COLUNAS_B = CENARIOS[nome_b]

SENSIBILIDADE = {
    'B (referência)': COLUNAS_B,
    'B sem TIPO_DIFICULDADE_COGNITIVA': COLUNAS_B + BLOCO_COGNITIVO[:1],
    'B sem bloco cognitivo': COLUNAS_B + BLOCO_COGNITIVO,
    'B sem bloco cognitivo e sem solidão': COLUNAS_B + BLOCO_COGNITIVO + SOLIDAO,
}

res_sens = {nome: treinar_avaliar(remover) for nome, remover in SENSIBILIDADE.items()}

tabela_sens = pd.DataFrame({nome: res[0] for nome, res in res_sens.items()}).T
tabela_sens['Variação do ROC-AUC vs. B'] = tabela_sens['ROC-AUC'] - tabela_sens.loc['B (referência)', 'ROC-AUC']
tabela_sens['Nº de variáveis'] = tabela_sens['Nº de variáveis'].astype(int)
print(f'Referência de acaso: ROC-AUC 0,5 e precisão {prevalencia:.1%} (prevalência)\n')
print('Top 5 variáveis de cada cenário:')
for nome, res in res_sens.items():
    print(f'  {nome}: {", ".join(res[3].head(5).index)}')
tabela_sens.round(4)

### Resultado do teste de sensibilidade

| Cenário | Variáveis | ROC-AUC | Precisão | Recall | F1 |
|---|---|---|---|---|---|
| B (referência) | 46 | 0,841 | 57,1% | 58,6% | 0,578 |
| sem `TIPO_DIFICULDADE_COGNITIVA` | 45 | 0,841 | 57,6% | 58,4% | 0,580 |
| sem bloco cognitivo | 43 | 0,827 | 58,0% | 58,4% | 0,582 |
| sem bloco cognitivo e sem solidão | 42 | **0,801** | 52,1% | 54,0% | 0,530 |

- **Tirar só o `TIPO`:** não muda nada. `QTD_DIFICULDADE_COGNITIVA` e `FREQUENCIA_FALHA_MEMORIA` assumem o papel dela.
- **Tirar o bloco cognitivo:** o ROC-AUC cai apenas 0,015. Precisão, recall e F1 ficam praticamente iguais. Portanto o bloco cognitivo não é o que sustenta o resultado.
- **Tirar também a solidão:** o ROC-AUC cai para 0,801 e a precisão para 52,1%, ainda mais que o dobro da prevalência (21,6%). Nesse cenário, `DOR_CRONICA` aparece entre as 5 variáveis mais importantes, junto com idade, suporte social, autoavaliação de saúde e renda.
- **Leitura para a hipótese:** mesmo sem as variáveis mais próximas de sintomas depressivos, o modelo continua bem acima do acaso. Isso reforça que **saúde física, funcionamento e contexto social** têm informação real sobre depressão, e que o resultado do cenário B não vem apenas de um efeito de sobreposição com sintomas.
- **Limite do teste:** ele retira só as variáveis que identificamos como suspeitas. Outras variáveis (por exemplo, suporte social) também podem se relacionar com sintomas. O teste mostra robustez, mas não prova ausência total de sobreposição.

# 10. Interpretação e conclusão

## Binário: a hipótese se sustenta, com desempenho menor que o do cenário A

| Métrica | A (com ansiedade) | B (sem sintomas mentais) | Acaso |
|---|---|---|---|
| ROC-AUC | 0,914 | **0,841** | 0,50 |
| Precisão (depressão) | 65,6% | **57,1%** | 21,6% (prevalência) |
| Recall (depressão) | 74,9% | **58,6%** | - |
| F1 (depressão) | 0,699 | **0,578** | - |

- Retirar as variáveis de ansiedade custa cerca de 0,07 de ROC-AUC e 16 pontos de recall. Isso era esperado, já que ansiedade e depressão são fortemente comórbidas (no cenário A, 9 das 10 variáveis mais importantes são de ansiedade ou cognição).
- Mesmo assim, **sem nenhum sintoma de saúde mental, o modelo fica 0,34 de ROC-AUC acima do acaso e a precisão é 2,6 vezes a prevalência** (57,1% contra 21,6%). Ou seja, dados físicos, funcionais e de contexto social **carregam informação real sobre depressão**.
- As variáveis mais importantes no cenário B são idade, dificuldade cognitiva, solidão, suporte social, autoavaliação de saúde, falha de memória, renda/pobreza e dor crônica. O sinal físico vem sobretudo de dor, limitações e autoavaliação de saúde, e não do diagnóstico de doenças clássicas. Isso é coerente com o estudo de associação em `02 Análise exploratória/07_associacao_depressao_variaveis_nao_mentais_2025.ipynb`.
- **Ressalva:** solidão, memória e dificuldade cognitiva podem se sobrepor aos próprios sintomas depressivos. Para sustentar a tese "dados físicos", vale destacar dor crônica, limitações funcionais e autoavaliação de saúde.
- A matriz ponderada (projeção populacional) serve só para a análise final e para estimar o volume de erros na população. O desempenho do modelo deve ser lido pelas métricas e pela matriz em contagem de registros.

## Multinível: informativo, mas pior que o binário para o objetivo do projeto

- Com 4 níveis, o cenário B tem ROC-AUC (OvR) de 0,826 e F1 macro de 0,447. O recall é bom no nível 1 (86%) e fraco nos níveis 3 e 4 (13% e 33%), por causa da raridade desses níveis e da dificuldade de separar níveis vizinhos (seção 6).

## Ajuste de hiperparâmetros

- **Binário:** a busca escolheu exatamente a configuração que já usávamos (300 árvores, profundidade 25, `log2`), então não houve ganho. Isso indica que o **limite está na informação disponível nas variáveis, não nos hiperparâmetros**.
- **Multinível:** o ROC-AUC ficou igual (0,826 contra 0,825) e o F1 macro caiu um pouco (0,447 para 0,440). A configuração ajustada recupera mais casos dos níveis 3 e 4 (recall de 19% e 41%), à custa do nível 1. Se o foco for detectar os casos mais graves, ela é preferível; para uma visão equilibrada, a configuração base basta.

## Teste de sensibilidade (seção 9)

- Retirar o bloco cognitivo (tipo e quantidade de dificuldade e frequência de falha de memória) reduz o ROC-AUC do cenário B de 0,841 para 0,827. Retirar também a solidão leva a 0,801, ainda bem acima do acaso (0,5).
- Portanto, a hipótese **não depende** das variáveis que mais se confundem com sintomas depressivos. Sem elas, os destaques passam a ser idade, suporte social, autoavaliação de saúde, renda e dor crônica.

## Limitações

- O resultado vale para registros completos (22.084 de 24.215); quem não respondeu a alguma pergunta ficou de fora.
- Métricas ponderadas pela amostra, com um único split treino/teste. Uma validação cruzada daria uma estimativa mais estável.
- Os modelos mostram **associação preditiva**, não causalidade.